# 01. Dataset Download & Preparation

Ноутбук подготавливает **отдельные** датасеты для трёх задач:

1. `view_dataset` — валидация ракурса обязательных фото  
2. `quality_dataset` — бинарный quality gate (`accept / reject`)  
3. `CarDD` raw — исходник для последующей конвертации в YOLO-seg

Что важно:
- старый multitask merge `viewpoint + quality` здесь **не используется**;
- все пути в проекте сохраняются в `ml/...`;
- split делается **до** аугментаций и до synthetic-генерации.

## Что будет создано

- `ml/data/CarDD`
- `ml/data/CompCars`
- `ml/data/KADID-10k` *(опционально, как reference по distortion-типам)*
- `ml/data/imagenet_other_invalid` *(небольшой subset non-car изображений из ImageNet-1k через Hugging Face, если включено авто-скачивание)*
- `ml/data/view_dataset/{train,val,test}/{front_valid,rear_valid,side_valid,angled_invalid,other_invalid}`
- `ml/data/quality_dataset/{train,val,test}/{accept,reject}`
- `ml/data/manifests/*.csv`


# ВАЖНО

Этот ноутбук **не использует ImageNet / Hugging Face**.

Класс `other_invalid` собирается только из:
- `ml/data/user_invalid_photos` — если ты положишь туда реальные невалидные фото;
- synthetic crop-ов из валидных и angled full-car изображений.

Если папка `user_invalid_photos` пустая, ноутбук всё равно отработает и доберёт `other_invalid` crop-ами.


In [1]:
# ============================================================
# НАСТРОЙКИ ВНЕШНИХ ТОКЕНОВ НЕ НУЖНЫ
# ============================================================
# В этой версии ноутбука Hugging Face / ImageNet полностью убраны.
# Никаких токенов задавать не нужно.
# ============================================================

print("HF / ImageNet disabled in this notebook version.")


HF / ImageNet disabled in this notebook version.


In [2]:
!pip install -q gdown opencv-python-headless pillow numpy pandas pyyaml tqdm scikit-learn matplotlib

In [3]:

import os
import sys
import json
import math
import csv
import random
import shutil
import subprocess
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance
from tqdm.auto import tqdm
import urllib.request


In [4]:

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    markers = ["pyproject.toml", "requirements.txt", ".git"]
    for cand in candidates:
        if any((cand / m).exists() for m in markers):
            return cand
    # notebooks обычно лежат в project_root/notebooks
    if start.parent.exists():
        return start.parent
    return start

PROJECT_ROOT = find_project_root(Path("."))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / "ml" / "data"
MANIFESTS_DIR = DATA_ROOT / "manifests"
QUALITY_VIEW_ROOT = PROJECT_ROOT / "ml" / "quality_view"

CARDD_DIR = DATA_ROOT / "CarDD"
COMPCARS_DIR = DATA_ROOT / "CompCars"
KADID_DIR = DATA_ROOT / "KADID-10k"   # optional reference only
VIEW_DATASET_DIR = DATA_ROOT / "view_dataset"
QUALITY_DATASET_DIR = DATA_ROOT / "quality_dataset"

USER_INVALID_DIR = DATA_ROOT / "user_invalid_photos"              # ручные невалидные фото

for p in [
    DATA_ROOT, MANIFESTS_DIR, QUALITY_VIEW_ROOT,
    CARDD_DIR, COMPCARS_DIR, KADID_DIR,
    VIEW_DATASET_DIR, QUALITY_DATASET_DIR,
    USER_INVALID_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)


PROJECT_ROOT = /Users/versache-karinndzi/Downloads/car-inspection-assistant
DATA_ROOT    = /Users/versache-karinndzi/Downloads/car-inspection-assistant/ml/data


## Параметры подготовки датасетов

Числа выбраны так, чтобы датасеты были **достаточно большими**, но при этом оставались реалистичными для обучения на MacBook.

In [5]:
CFG = {
    "split": {"train": 0.70, "val": 0.15, "test": 0.15},
    "view": {
        "target_per_class": 2200,
        "other_invalid_target": 2200,
        "max_user_invalid": 1200,
        "max_crop_invalid": 2200,
        "angled_crop_fraction": 0.18,
        "image_size": 256,
    },
    "quality": {
        "max_base_images": 3000,
        "target_accept_total": 5200,
        "target_reject_total": 5200,
        "image_size": 256,
        "accept_threshold": 0.55,
        "reject_threshold": 1.00,
        "weak_single_threshold": 0.45,
        "strong_single_threshold": 0.78,
    },
    "download": {
        "download_kadid_reference": False,
    }
}

CFG_PATH = QUALITY_VIEW_ROOT / "configs" / "dataset_build.yaml"
CFG_PATH.parent.mkdir(parents=True, exist_ok=True)
CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved config to", CFG_PATH)


Saved config to /Users/versache-karinndzi/Downloads/car-inspection-assistant/ml/quality_view/configs/dataset_build.yaml


In [6]:

def explore_dir(path: Path, max_depth: int = 2, prefix: str = ""):
    path = Path(path)
    if not path.exists():
        print(f"{path} [missing]")
        return
    print(path)
    base_depth = len(path.parts)
    for item in sorted(path.rglob("*")):
        depth = len(item.parts) - base_depth
        if depth > max_depth:
            continue
        indent = "  " * depth
        suffix = "/" if item.is_dir() else ""
        print(f"{indent}{item.name}{suffix}")

def split_list(items, ratios=(0.7, 0.15, 0.15), seed=42):
    items = list(items)
    rnd = random.Random(seed)
    rnd.shuffle(items)
    n = len(items)
    n_train = int(n * ratios[0])
    n_val = int(n * ratios[1])
    return {
        "train": items[:n_train],
        "val": items[n_train:n_train+n_val],
        "test": items[n_train+n_val:],
    }

def ensure_empty_class_dirs(root_dir: Path, splits, classes):
    for split in splits:
        for cls in classes:
            d = root_dir / split / cls
            if d.exists():
                shutil.rmtree(d)
            d.mkdir(parents=True, exist_ok=True)

## 1. Скачивание / проверка сырого CarDD

CarDD обычно требует ручного получения ссылки. Ноутбук сохраняет тот же путь `ml/data/CarDD`.

In [7]:

import zipfile
import urllib.request

CARDD_URL = ""  # если есть прямая ссылка на zip, вставьте сюда
CARDD_ZIP = CARDD_DIR / "CarDD_release.zip"

def cardd_ready(cardd_dir: Path) -> bool:
    return any(cardd_dir.rglob("instances_train2017.json")) or any(cardd_dir.rglob("CarDD_COCO"))

def try_download_cardd(url: str, dst_zip: Path, dst_dir: Path):
    if cardd_ready(dst_dir):
        print("[CarDD] already ready")
        return True
    if not url:
        print("[CarDD] automatic URL not specified.")
        print("  Поместите распакованный CarDD в:", dst_dir)
        return False
    try:
        print("[CarDD] downloading...")
        urllib.request.urlretrieve(url, dst_zip)
        with zipfile.ZipFile(dst_zip, "r") as zf:
            zf.extractall(dst_dir)
        return cardd_ready(dst_dir)
    except Exception as e:
        print("[CarDD] download failed:", e)
        return False

_ = try_download_cardd(CARDD_URL, CARDD_ZIP, CARDD_DIR)
print("CarDD ready:", cardd_ready(CARDD_DIR))

[CarDD] already ready
CarDD ready: True


## 2. Скачивание / проверка CompCars

Сохраняем прежний проектный путь: `ml/data/CompCars`.

In [8]:

import gdown

COMPCARS_FOLDER_URL = "https://drive.google.com/drive/folders/18EunmjOJsbE5Lh9zA0cZ4wKV6Um46dkg"
COMPCARS_PASSWORD = "d89551fd190e38"

def compcars_is_ready(root_dir: Path) -> bool:
    image_dir = root_dir / "data" / "image"
    label_dir = root_dir / "data" / "label"
    return image_dir.exists() and label_dir.exists() and any(image_dir.rglob("*.jpg"))

def find_compcars_parts_dir(root_dir: Path):
    for data_zip in root_dir.rglob("data.zip"):
        return data_zip.parent
    return None

def download_compcars():
    if compcars_is_ready(COMPCARS_DIR):
        print("[CompCars] already ready")
        return True

    parts_dir = find_compcars_parts_dir(COMPCARS_DIR)
    if parts_dir is None:
        print("[CompCars] downloading multipart archive folder...")
        try:
            gdown.download_folder(COMPCARS_FOLDER_URL, output=str(COMPCARS_DIR), quiet=False)
        except Exception as e:
            print("[CompCars] download error:", e)
            print("Manual fallback: https://mmlab.ie.cuhk.edu.hk/datasets/comp_cars/")
            return False
        parts_dir = find_compcars_parts_dir(COMPCARS_DIR)

    if parts_dir is None:
        print("[CompCars] data.zip not found after download.")
        return False

    data_zip = parts_dir / "data.zip"
    combined_zip = COMPCARS_DIR / "combined.zip"

    if shutil.which("zip") is None or shutil.which("unzip") is None:
        print("Install `zip` and `unzip`, then run manually:")
        print(f"zip -F {data_zip} --out {combined_zip}")
        print(f"unzip -o -P {COMPCARS_PASSWORD} {combined_zip} -d {COMPCARS_DIR}")
        return False

    try:
        subprocess.run(["zip", "-F", str(data_zip), "--out", str(combined_zip)], cwd=str(parts_dir), check=True)
        subprocess.run(["unzip", "-o", "-P", COMPCARS_PASSWORD, str(combined_zip), "-d", str(COMPCARS_DIR)], check=True)
    except subprocess.CalledProcessError as e:
        print("[CompCars] extraction failed:", e)
        return False

    return compcars_is_ready(COMPCARS_DIR)

_ = download_compcars()
print("CompCars ready:", compcars_is_ready(COMPCARS_DIR))

[CompCars] already ready
CompCars ready: True


## 3. Опционально: KADID как reference по искажениям

In [9]:
import zipfile
import urllib.request

KADID_URL = "https://datasets.vqa.mmsp-kn.de/archives/kadid10k.zip"
KADID_ZIP = KADID_DIR / "kadid10k.zip"

def kadid_ready(root_dir: Path) -> bool:
    return any(root_dir.rglob("*.png"))

if CFG["download"]["download_kadid_reference"] and not kadid_ready(KADID_DIR):
    try:
        print("[KADID] downloading...")
        urllib.request.urlretrieve(KADID_URL, KADID_ZIP)
        with zipfile.ZipFile(KADID_ZIP, "r") as zf:
            zf.extractall(KADID_DIR)
    except Exception as e:
        print("[KADID] download failed:", e)

print("KADID ready:", kadid_ready(KADID_DIR))
print("ImageNet/Hugging Face disabled. other_invalid will use user_invalid_photos + synthetic crops.")


KADID ready: False
ImageNet/Hugging Face disabled. other_invalid will use user_invalid_photos + synthetic crops.


## 4. Подготовка `view_dataset`

Классы:
- `front_valid`
- `rear_valid`
- `side_valid`
- `angled_invalid`
- `other_invalid`

Где:
- `front / rear / side` и `front-side + rear-side` берутся из CompCars full-car
- `other_invalid` собирается из:
  - `ml/data/user_invalid_photos`
  - `ml/data/imagenet_other_invalid`
  - synthetic crop-ов из валидных full-car изображений

`CompCars parts` для сборки `other_invalid` больше не требуются.


In [10]:
VIEW_CLASSES = ["front_valid", "rear_valid", "side_valid", "angled_invalid", "other_invalid"]
VIEWPOINT_ID_TO_CLASS = {
    1: "front_valid",
    2: "rear_valid",
    3: "side_valid",
    4: "angled_invalid",
    5: "angled_invalid",
}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def parse_compcars_fullcar_samples(compcars_dir: Path):
    image_root = compcars_dir / "data" / "image"
    label_root = compcars_dir / "data" / "label"
    if not image_root.exists() or not label_root.exists():
        raise FileNotFoundError("CompCars full-car structure data/image + data/label not found")

    samples = []
    for txt in tqdm(sorted(label_root.rglob("*.txt")), desc="Parse CompCars labels"):
        try:
            first_line = txt.read_text(encoding="utf-8", errors="ignore").splitlines()[0].strip()
            vp_id = int(first_line)
        except Exception:
            continue
        if vp_id not in VIEWPOINT_ID_TO_CLASS:
            continue
        rel = txt.relative_to(label_root).with_suffix(".jpg")
        img = image_root / rel
        if not img.exists():
            alt = image_root / rel.with_suffix(".png")
            if alt.exists():
                img = alt
            else:
                continue
        samples.append({
            "image_path": img,
            "vp_id": vp_id,
            "class_name": VIEWPOINT_ID_TO_CLASS[vp_id],
            "source": "compcars_fullcar",
        })
    return samples

def list_images(root: Path):
    if not root.exists():
        return []
    return sorted([p for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS])

def collect_user_invalid_images(user_invalid_dir: Path, max_items: int):
    imgs = list_images(user_invalid_dir)
    rnd = random.Random(SEED)
    rnd.shuffle(imgs)
    return imgs[:max_items]

def copy_rgb(src: Path, dst: Path, size: int | None = None):
    img = Image.open(src).convert("RGB")
    if size:
        img.thumbnail((size, size))
        canvas = Image.new("RGB", (size, size), (0, 0, 0))
        x = (size - img.width) // 2
        y = (size - img.height) // 2
        canvas.paste(img, (x, y))
        img = canvas
    img.save(dst, quality=95)

def make_partial_crop(src: Path, dst: Path, size: int, rnd: random.Random):
    img = Image.open(src).convert("RGB")
    w, h = img.size
    if w < 40 or h < 40:
        return False

    mode = rnd.choice(["small_crop", "thin_crop", "corner_crop", "tiny_car"])
    if mode == "small_crop":
        crop_w = int(w * rnd.uniform(0.18, 0.45))
        crop_h = int(h * rnd.uniform(0.18, 0.45))
        x0 = rnd.randint(0, max(0, w - crop_w))
        y0 = rnd.randint(0, max(0, h - crop_h))
        crop = img.crop((x0, y0, min(w, x0 + crop_w), min(h, y0 + crop_h)))
    elif mode == "thin_crop":
        if rnd.random() < 0.5:
            crop_w = int(w * rnd.uniform(0.12, 0.22))
            x0 = rnd.randint(0, max(0, w - crop_w))
            crop = img.crop((x0, 0, min(w, x0 + crop_w), h))
        else:
            crop_h = int(h * rnd.uniform(0.12, 0.22))
            y0 = rnd.randint(0, max(0, h - crop_h))
            crop = img.crop((0, y0, w, min(h, y0 + crop_h)))
    elif mode == "corner_crop":
        crop_w = int(w * rnd.uniform(0.20, 0.35))
        crop_h = int(h * rnd.uniform(0.20, 0.35))
        corners = [
            (0, 0),
            (max(0, w - crop_w), 0),
            (0, max(0, h - crop_h)),
            (max(0, w - crop_w), max(0, h - crop_h)),
        ]
        x0, y0 = rnd.choice(corners)
        crop = img.crop((x0, y0, min(w, x0 + crop_w), min(h, y0 + crop_h)))
    else:
        scale = rnd.uniform(0.14, 0.30)
        nw, nh = max(24, int(w * scale)), max(24, int(h * scale))
        small = img.resize((nw, nh))
        canvas = Image.new("RGB", (w, h), (0, 0, 0))
        x0 = rnd.randint(0, max(0, w - nw))
        y0 = rnd.randint(0, max(0, h - nh))
        canvas.paste(small, (x0, y0))
        crop = canvas

    crop.thumbnail((size, size))
    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    x = (size - crop.width) // 2
    y = (size - crop.height) // 2
    canvas.paste(crop, (x, y))
    dst.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(dst, quality=95)
    return True

def make_angled_crop(src: Path, dst: Path, size: int, rnd: random.Random):
    img = Image.open(src).convert("RGB")
    w, h = img.size
    if w < 60 or h < 60:
        return False

    crop_w = int(w * rnd.uniform(0.55, 0.82))
    crop_h = int(h * rnd.uniform(0.55, 0.82))

    anchor = rnd.choice(["left", "right", "top", "bottom", "corner"])
    if anchor == "left":
        x0 = 0
        y0 = rnd.randint(0, max(0, h - crop_h))
    elif anchor == "right":
        x0 = max(0, w - crop_w)
        y0 = rnd.randint(0, max(0, h - crop_h))
    elif anchor == "top":
        x0 = rnd.randint(0, max(0, w - crop_w))
        y0 = 0
    elif anchor == "bottom":
        x0 = rnd.randint(0, max(0, w - crop_w))
        y0 = max(0, h - crop_h)
    else:
        corners = [
            (0, 0),
            (max(0, w - crop_w), 0),
            (0, max(0, h - crop_h)),
            (max(0, w - crop_w), max(0, h - crop_h)),
        ]
        x0, y0 = rnd.choice(corners)

    crop = img.crop((x0, y0, min(w, x0 + crop_w), min(h, y0 + crop_h)))
    crop.thumbnail((size, size))
    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    x = (size - crop.width) // 2
    y = (size - crop.height) // 2
    canvas.paste(crop, (x, y))
    dst.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(dst, quality=95)
    return True


In [11]:

fullcar_samples = parse_compcars_fullcar_samples(COMPCARS_DIR)
print("Full-car samples:", len(fullcar_samples))
print(Counter(s["class_name"] for s in fullcar_samples))

Parse CompCars labels:   0%|          | 0/89384 [00:00<?, ?it/s]

Full-car samples: 88860
Counter({'angled_invalid': 53083, 'side_valid': 15198, 'front_valid': 11923, 'rear_valid': 8656})


In [12]:
# 1) Балансируем валидные и angled-классы из full-car
fullcar_by_class = defaultdict(list)
for s in fullcar_samples:
    fullcar_by_class[s["class_name"]].append(s)

for cls in ["front_valid", "rear_valid", "side_valid", "angled_invalid"]:
    random.Random(SEED).shuffle(fullcar_by_class[cls])

target = CFG["view"]["target_per_class"]
splits = list(CFG["split"].keys())
ensure_empty_class_dirs(VIEW_DATASET_DIR, splits, VIEW_CLASSES)

view_manifest = []
valid_manifest_by_split = defaultdict(list)
angled_manifest_by_split = defaultdict(list)

# 1a) valid classes copy as-is
for cls in ["front_valid", "rear_valid", "side_valid"]:
    selected = fullcar_by_class[cls][:target]
    split_map = split_list(selected, ratios=(CFG["split"]["train"], CFG["split"]["val"], CFG["split"]["test"]), seed=SEED)

    for split, rows in split_map.items():
        for idx, row in enumerate(tqdm(rows, desc=f"Copy {split}/{cls}")):
            dst = VIEW_DATASET_DIR / split / cls / f"{idx:06d}.jpg"
            copy_rgb(row["image_path"], dst, size=CFG["view"]["image_size"])
            view_manifest.append({
                "split": split,
                "class_name": cls,
                "dst_path": str(dst),
                "src_path": str(row["image_path"]),
                "source": row["source"],
                "kind": "fullcar",
            })
            valid_manifest_by_split[split].append(Path(row["image_path"]))

# 1b) angled_invalid: mix of raw angled full-car + angled-preserving crops
angled_selected = fullcar_by_class["angled_invalid"][:target]
angled_split_map = split_list(angled_selected, ratios=(CFG["split"]["train"], CFG["split"]["val"], CFG["split"]["test"]), seed=SEED)
angled_crop_fraction = CFG["view"].get("angled_crop_fraction", 0.18)

for split, rows in angled_split_map.items():
    split_target = len(rows)
    crop_target = min(split_target, max(0, int(round(split_target * angled_crop_fraction))))
    raw_target = split_target - crop_target

    for idx, row in enumerate(tqdm(rows[:raw_target], desc=f"Copy {split}/angled_invalid raw")):
        dst = VIEW_DATASET_DIR / split / "angled_invalid" / f"{idx:06d}.jpg"
        copy_rgb(row["image_path"], dst, size=CFG["view"]["image_size"])
        view_manifest.append({
            "split": split,
            "class_name": "angled_invalid",
            "dst_path": str(dst),
            "src_path": str(row["image_path"]),
            "source": row["source"],
            "kind": "fullcar",
        })
        angled_manifest_by_split[split].append(Path(row["image_path"]))

    rnd = random.Random(SEED + {"train": 101, "val": 103, "test": 107}[split])
    pool = [Path(r["image_path"]) for r in rows]
    counter = raw_target
    attempts = 0
    max_attempts = max(40, crop_target * 12)

    while counter < split_target and attempts < max_attempts and pool:
        src = rnd.choice(pool)
        dst = VIEW_DATASET_DIR / split / "angled_invalid" / f"{counter:06d}.jpg"
        ok = make_angled_crop(src, dst, size=CFG["view"]["image_size"], rnd=rnd)
        attempts += 1
        if ok:
            view_manifest.append({
                "split": split,
                "class_name": "angled_invalid",
                "dst_path": str(dst),
                "src_path": str(src),
                "source": "synthetic_angled_crop",
                "kind": "angled_crop",
            })
            angled_manifest_by_split[split].append(Path(src))
            counter += 1

# 2) other_invalid: user_invalid + synthetic crop invalids
user_invalid_imgs = collect_user_invalid_images(USER_INVALID_DIR, CFG["view"]["max_user_invalid"])

raw_other_invalid = [
    {"image_path": p, "source": "user_invalid"} for p in user_invalid_imgs
]

print("Raw other_invalid sources:")
print("  user_invalid =", len(user_invalid_imgs))
print("  synthetic crop fallback will fill the rest.")

other_splits = split_list(raw_other_invalid, ratios=(CFG["split"]["train"], CFG["split"]["val"], CFG["split"]["test"]), seed=SEED)
other_counter = {"train": 0, "val": 0, "test": 0}

other_target_total = CFG["view"].get("other_invalid_target", CFG["view"]["target_per_class"])
other_targets_per_split = {
    "train": int(other_target_total * CFG["split"]["train"]),
    "val": int(other_target_total * CFG["split"]["val"]),
    "test": other_target_total - int(other_target_total * CFG["split"]["train"]) - int(other_target_total * CFG["split"]["val"]),
}

# 2a) сначала копируем доступные raw invalid
for split, rows in other_splits.items():
    max_raw_for_split = min(len(rows), other_targets_per_split[split])
    for row in tqdm(rows[:max_raw_for_split], desc=f"Copy {split}/other_invalid raw"):
        dst = VIEW_DATASET_DIR / split / "other_invalid" / f"{other_counter[split]:06d}.jpg"
        copy_rgb(row["image_path"], dst, size=CFG["view"]["image_size"])
        view_manifest.append({
            "split": split,
            "class_name": "other_invalid",
            "dst_path": str(dst),
            "src_path": str(row["image_path"]),
            "source": row["source"],
            "kind": "raw_invalid",
        })
        other_counter[split] += 1

# 2b) добиваем до target synthetic crop-ами из валидных и angled изображений того же сплита
crop_source_by_split = defaultdict(list)
for split in ["train", "val", "test"]:
    crop_source_by_split[split].extend(valid_manifest_by_split[split])
    crop_source_by_split[split].extend(angled_manifest_by_split[split])

for split in ["train", "val", "test"]:
    pool = crop_source_by_split[split]
    if not pool:
        continue

    rnd = random.Random(SEED + {"train": 11, "val": 17, "test": 23}[split])
    attempts = 0
    max_attempts = max(50, CFG["view"]["max_crop_invalid"] * 12)

    while other_counter[split] < other_targets_per_split[split] and attempts < max_attempts:
        src = rnd.choice(pool)
        dst = VIEW_DATASET_DIR / split / "other_invalid" / f"{other_counter[split]:06d}.jpg"
        ok = make_partial_crop(src, dst, size=CFG["view"]["image_size"], rnd=rnd)
        attempts += 1
        if ok:
            view_manifest.append({
                "split": split,
                "class_name": "other_invalid",
                "dst_path": str(dst),
                "src_path": str(src),
                "source": "synthetic_crop",
                "kind": "crop_invalid",
            })
            other_counter[split] += 1

for split in ["train", "val", "test"]:
    print(f"{split} other_invalid: {other_counter[split]} / {other_targets_per_split[split]}")

view_manifest_df = pd.DataFrame(view_manifest)
view_manifest_path = MANIFESTS_DIR / "view_dataset_manifest.csv"
view_manifest_df.to_csv(view_manifest_path, index=False)
print("Saved view manifest:", view_manifest_path)
print(view_manifest_df.groupby(["split", "class_name"]).size().unstack(fill_value=0))


Copy train/front_valid:   0%|          | 0/1540 [00:00<?, ?it/s]

Copy val/front_valid:   0%|          | 0/330 [00:00<?, ?it/s]

Copy test/front_valid:   0%|          | 0/330 [00:00<?, ?it/s]

Copy train/rear_valid:   0%|          | 0/1540 [00:00<?, ?it/s]

Copy val/rear_valid:   0%|          | 0/330 [00:00<?, ?it/s]

...

Raw other_invalid sources:
  user_invalid = 0
  synthetic crop fallback will fill the rest.


Copy train/other_invalid raw: 0it [00:00, ?it/s]

Copy val/other_invalid raw: 0it [00:00, ?it/s]

Copy test/other_invalid raw: 0it [00:00, ?it/s]

train other_invalid: 1540 / 1540
val other_invalid: 330 / 330
test other_invalid: 330 / 330
Saved view manifest: /Users/versache-karinndzi/Downloads/car-inspection-assistant/ml/data/manifests/view_dataset_manifest.csv
class_name  angled_invalid  front_valid  other_invalid  rear_valid  side_valid
split                                                                         
test                   330          330            330         330         330
train                 1540         1540           1540        1540        1540
val                    330          330            330         330         330


## 5. Подготовка `quality_dataset`

Классы:
- `accept`
- `reject`

База:
- full-car валидные фото из CompCars
- фотографии машин из CarDD

Разметка автоматическая:
- `reject`, если хотя бы одно искажение сильное **или** сумма нескольких умеренных превышает порог
- `accept`, если искажения отсутствуют/слабые
- серую зону пропускаем

In [13]:

QUALITY_CLASSES = ["accept", "reject"]

def collect_cardd_images(cardd_dir: Path):
    imgs = [p for p in cardd_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    # Отфильтруем служебные папки labels/annotations
    imgs = [p for p in imgs if "label" not in str(p.parent).lower() and "annotation" not in str(p.parent).lower()]
    return sorted(imgs)

def build_quality_base_pool():
    pool = []
    # valid full-car from CompCars
    for row in fullcar_samples:
        if row["class_name"] in {"front_valid", "rear_valid", "side_valid"}:
            pool.append({"image_path": row["image_path"], "source": "compcars_fullcar"})
    # car-domain from CarDD
    for p in collect_cardd_images(CARDD_DIR):
        pool.append({"image_path": p, "source": "cardd"})
    # dedup by resolved path
    seen = set()
    uniq = []
    for row in pool:
        rp = str(Path(row["image_path"]).resolve())
        if rp in seen:
            continue
        seen.add(rp)
        uniq.append(row)
    random.Random(SEED).shuffle(uniq)
    max_base_images = CFG["quality"].get("max_base_images", len(uniq))
    return uniq[:max_base_images]

base_pool = build_quality_base_pool()
print("Quality base pool:", len(base_pool))
print(Counter(r["source"] for r in base_pool))


Quality base pool: 3000
Counter({'compcars_fullcar': 2099, 'cardd': 901})


In [14]:

def pil_to_cv(img: Image.Image):
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

def cv_to_pil(arr):
    return Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB))

def apply_motion_blur(img: Image.Image, ksize: int):
    arr = pil_to_cv(img)
    kernel = np.zeros((ksize, ksize), dtype=np.float32)
    kernel[ksize // 2, :] = 1.0 / ksize
    arr = cv2.filter2D(arr, -1, kernel)
    return cv_to_pil(arr)

def apply_gaussian_blur(img: Image.Image, sigma: float):
    arr = pil_to_cv(img)
    arr = cv2.GaussianBlur(arr, (0, 0), sigmaX=sigma, sigmaY=sigma)
    return cv_to_pil(arr)

def apply_noise(img: Image.Image, sigma: float):
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, sigma, arr.shape).astype(np.float32)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def apply_darken(img: Image.Image, factor: float):
    return ImageEnhance.Brightness(img).enhance(factor)

def apply_overexpose(img: Image.Image, factor: float):
    return ImageEnhance.Brightness(img).enhance(factor)

def apply_downscale_upscale(img: Image.Image, scale: float):
    w, h = img.size
    nw, nh = max(16, int(w * scale)), max(16, int(h * scale))
    low = img.resize((nw, nh), Image.BILINEAR)
    return low.resize((w, h), Image.BILINEAR)

def apply_jpeg_compression(img: Image.Image, quality: int):
    import io
    buff = io.BytesIO()
    img.save(buff, format="JPEG", quality=quality)
    buff.seek(0)
    return Image.open(buff).convert("RGB")

DISTORTION_WEIGHTS = {
    "gaussian_blur": 0.50,
    "motion_blur": 0.55,
    "darken": 0.45,
    "overexpose": 0.45,
    "noise": 0.25,
    "downscale": 0.30,
    "jpeg": 0.20,
}

def sample_distortions(target_label: str, rnd: random.Random):
    if target_label == "accept":
        n = rnd.choice([0, 1, 1, 2])
        low, high = 0.05, 0.38
    else:
        n = rnd.choice([1, 1, 2, 2, 3])
        low, high = 0.35, 1.0

    names = list(DISTORTION_WEIGHTS.keys())
    chosen = rnd.sample(names, k=n) if n else []
    severities = {name: rnd.uniform(low, high) for name in chosen}
    return severities

def score_distortions(severities: dict):
    if not severities:
        return 0.0, 0.0
    max_sev = max(severities.values())
    weighted_sum = sum(DISTORTION_WEIGHTS[k] * v for k, v in severities.items())
    return max_sev, weighted_sum

def apply_distortions(img: Image.Image, severities: dict):
    out = img.copy()
    for name, sev in severities.items():
        if name == "gaussian_blur":
            out = apply_gaussian_blur(out, sigma=0.6 + 3.0 * sev)
        elif name == "motion_blur":
            k = int(3 + sev * 10)
            if k % 2 == 0:
                k += 1
            out = apply_motion_blur(out, ksize=max(3, k))
        elif name == "darken":
            out = apply_darken(out, factor=max(0.12, 1.0 - 0.8 * sev))
        elif name == "overexpose":
            out = apply_overexpose(out, factor=1.0 + 1.35 * sev)
        elif name == "noise":
            out = apply_noise(out, sigma=4 + 28 * sev)
        elif name == "downscale":
            out = apply_downscale_upscale(out, scale=max(0.12, 1.0 - 0.78 * sev))
        elif name == "jpeg":
            out = apply_jpeg_compression(out, quality=max(8, int(95 - 80 * sev)))
    return out

def auto_label_from_severities(severities: dict, cfg):
    max_sev, weighted_sum = score_distortions(severities)
    if max_sev >= cfg["strong_single_threshold"] or weighted_sum >= cfg["reject_threshold"]:
        return "reject"
    if max_sev <= cfg["weak_single_threshold"] and weighted_sum <= cfg["accept_threshold"]:
        return "accept"
    return "skip"

def save_square(img: Image.Image, dst: Path, size: int):
    img = img.convert("RGB")
    img.thumbnail((size, size))
    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    canvas.paste(img, ((size - img.width)//2, (size - img.height)//2))
    canvas.save(dst, quality=95)


In [15]:

quality_manifest = []
ensure_empty_class_dirs(QUALITY_DATASET_DIR, ["train", "val", "test"], QUALITY_CLASSES)

base_splits = split_list(base_pool, ratios=(CFG["split"]["train"], CFG["split"]["val"], CFG["split"]["test"]), seed=SEED)

targets_per_split = {
    "train": {
        "accept": int(CFG["quality"]["target_accept_total"] * CFG["split"]["train"]),
        "reject": int(CFG["quality"]["target_reject_total"] * CFG["split"]["train"]),
    },
    "val": {
        "accept": int(CFG["quality"]["target_accept_total"] * CFG["split"]["val"]),
        "reject": int(CFG["quality"]["target_reject_total"] * CFG["split"]["val"]),
    },
    "test": {
        "accept": CFG["quality"]["target_accept_total"] - int(CFG["quality"]["target_accept_total"] * CFG["split"]["train"]) - int(CFG["quality"]["target_accept_total"] * CFG["split"]["val"]),
        "reject": CFG["quality"]["target_reject_total"] - int(CFG["quality"]["target_reject_total"] * CFG["split"]["train"]) - int(CFG["quality"]["target_reject_total"] * CFG["split"]["val"]),
    }
}

for split, base_rows in base_splits.items():
    counters = {"accept": 0, "reject": 0}
    rnd = random.Random(SEED)

    # 1) сначала оригиналы как accept
    for row in tqdm(base_rows, desc=f"{split}: seed accept originals"):
        if counters["accept"] >= targets_per_split[split]["accept"]:
            break
        try:
            img = Image.open(row["image_path"]).convert("RGB")
            dst = QUALITY_DATASET_DIR / split / "accept" / f"{counters['accept']:06d}.jpg"
            save_square(img, dst, CFG["quality"]["image_size"])
            quality_manifest.append({
                "split": split,
                "label": "accept",
                "dst_path": str(dst),
                "src_path": str(row["image_path"]),
                "source": row["source"],
                "distortions": "{}",
            })
            counters["accept"] += 1
        except Exception:
            continue

    # 2) затем добиваем accept/reject synthetic-версиями
    attempts = 0
    while counters["accept"] < targets_per_split[split]["accept"] or counters["reject"] < targets_per_split[split]["reject"]:
        attempts += 1
        if attempts > (targets_per_split[split]["accept"] + targets_per_split[split]["reject"]) * 15:
            break

        row = rnd.choice(base_rows)
        target_label = "accept" if counters["accept"] < targets_per_split[split]["accept"] and (counters["accept"] <= counters["reject"]) else "reject"
        try:
            base_img = Image.open(row["image_path"]).convert("RGB")
        except Exception:
            continue

        severities = sample_distortions(target_label, rnd)
        auto_label = auto_label_from_severities(severities, CFG["quality"])

        if auto_label == "skip" or auto_label != target_label:
            continue

        out = apply_distortions(base_img, severities)
        dst = QUALITY_DATASET_DIR / split / target_label / f"{counters[target_label]:06d}.jpg"
        save_square(out, dst, CFG["quality"]["image_size"])

        quality_manifest.append({
            "split": split,
            "label": target_label,
            "dst_path": str(dst),
            "src_path": str(row["image_path"]),
            "source": row["source"],
            "distortions": json.dumps(severities),
        })
        counters[target_label] += 1

quality_manifest_df = pd.DataFrame(quality_manifest)
quality_manifest_path = MANIFESTS_DIR / "quality_dataset_manifest.csv"
quality_manifest_df.to_csv(quality_manifest_path, index=False)
print("Saved quality manifest:", quality_manifest_path)
print(quality_manifest_df.groupby(["split", "label"]).size().unstack(fill_value=0))

train: seed accept originals:   0%|          | 0/2100 [00:00<?, ?it/s]

val: seed accept originals:   0%|          | 0/450 [00:00<?, ?it/s]

test: seed accept originals:   0%|          | 0/450 [00:00<?, ?it/s]

Saved quality manifest: /Users/versache-karinndzi/Downloads/car-inspection-assistant/ml/data/manifests/quality_dataset_manifest.csv
label  accept  reject
split                
test      781     781
train    3639    3639
val       780     780


## 6. Быстрая проверка структуры и следующих шагов

In [16]:

print("\nView dataset:")
explore_dir(VIEW_DATASET_DIR, max_depth=2)

print("\nQuality dataset:")
explore_dir(QUALITY_DATASET_DIR, max_depth=2)

print("\nNext notebooks:")
print("  02_train_quality_view.ipynb     -> train view validation + quality gate (раздельно, в одном файле для совместимости)")
print("  01_cardd_to_yolo_only.ipynb     -> convert CarDD to YOLO-seg with all 6 classes")
print("  03_train_damage_segmentation.ipynb -> train segmentation model")
print("  04_evaluate_pipeline.ipynb      -> end-to-end evaluation")


View dataset:
/Users/versache-karinndzi/Downloads/car-inspection-assistant/ml/data/view_dataset
  test/
    angled_invalid/
    front_valid/
    other_invalid/
    rear_valid/
    side_valid/
  train/
    angled_invalid/
    front_valid/
    other_invalid/
    rear_valid/
    side_valid/
  val/
    angled_invalid/
    front_valid/
    other_invalid/
    rear_valid/
    side_valid/

Quality dataset:
/Users/versache-karinndzi/Downloads/car-inspection-assistant/ml/data/quality_dataset
  test/
    accept/
    reject/
  train/
    accept/
    reject/
  val/
    accept/
    reject/

Next notebooks:
  02_train_quality_view.ipynb     -> train view validation + quality gate (раздельно, в одном файле для совместимости)
  01_cardd_to_yolo_only.ipynb     -> convert CarDD to YOLO-seg with all 6 classes
  03_train_damage_segmentation.ipynb -> train segmentation model
  04_evaluate_pipeline.ipynb      -> end-to-end evaluation
